# Audio Event Detection (Cry / Scream) Fine-Tuning (Colab)

Trains a small **log-mel CNN** to detect distress sounds, replacing the brittle
DSP heuristic (pitch + RMS thresholds) in `stages/audio_analysis.py`. A learned
classifier generalizes far better to real home audio.

Classes are auto-discovered from what you provide:
  baby_cry  (ESC-50 `crying_baby` + donate-a-cry corpus)
  other     (the rest of ESC-50 - the crucial negatives that stop false alarms)
  scream    (OPTIONAL - drop a scream dataset in SCREAM_ROOT to add this class)

**Tuny model, runs anywhere** (CPU-fine at inference). Feature caching is
resume-safe.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# =====================================================================
#  CONFIG  -  after you upload the whole Fine_tuning folder to
#  MyDrive/VisionAI/, these paths already point at this model folder.
#  (datasets live INSIDE it - see DATASETS.txt.)  "" = skip a source.
# =====================================================================
BASE = "/content/drive/MyDrive/VisionAI/Fine_tuning/05_audio_events"

ESC50_ROOT      = BASE + "/ESC-50"            # has audio/ and meta/esc50.csv
DONATEACRY_ROOT = BASE + "/donateacry-corpus" # any .wav under here -> baby_cry
SCREAM_ROOT     = BASE + "/scream/Converted_Separately/scream"                          # OPTIONAL: .wav/.mp3 -> scream

WORK_DIR = BASE + "/_work"

# =====================================================================
#  HYPERPARAMETERS  (audio front-end MUST match the app at inference)
# =====================================================================
SR        = 16000
DURATION  = 4.0           # seconds per clip (pad/truncate)
N_FFT     = 1024
HOP       = 512
N_MELS    = 64
MAX_OTHER = 800           # cap ESC-50 negatives so it stays balanced/fast

BATCH_SIZE = 64
EPOCHS     = 60           # max per fold; early stopping ends sooner
LR         = 1e-3
WEIGHT_DECAY = 1e-4
SEED       = 42

import os
N_SAMPLES = int(SR * DURATION)
os.makedirs(WORK_DIR, exist_ok=True)
CACHE_DIR = os.path.join(WORK_DIR, "logmel_cache")
os.makedirs(CACHE_DIR, exist_ok=True)
print("Work dir:", WORK_DIR)

In [ ]:
!pip -q install librosa soundfile
import torch, subprocess
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print(subprocess.getoutput("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader"))

## Phase 1 - Gather clips + cache log-mel features
ESC-50's `meta/esc50.csv` maps each clip to a category; `crying_baby` becomes the
positive, everything else becomes `other`. donate-a-cry adds more cry variety.
Each clip -> a normalized log-mel `[N_MELS, T]` cached as `.npy` (resume-safe).

In [ ]:
import os, csv, glob, hashlib, random, numpy as np, librosa
from tqdm import tqdm
random.seed(SEED); np.random.seed(SEED)
AUDIO_EXT = ('.wav', '.ogg', '.flac', '.mp3')

def _list_audio(d):
    out = []
    for root, _, files in os.walk(d):
        for f in files:
            if f.lower().endswith(AUDIO_EXT): out.append(os.path.join(root, f))
    return out

samples = []   # (path, label_str)
if ESC50_ROOT and os.path.isdir(ESC50_ROOT):
    meta = os.path.join(ESC50_ROOT, "meta", "esc50.csv")
    audio = os.path.join(ESC50_ROOT, "audio")
    rows = list(csv.DictReader(open(meta)))
    cry = [r for r in rows if r["category"] == "crying_baby"]
    oth = [r for r in rows if r["category"] != "crying_baby"]
    random.shuffle(oth); oth = oth[:MAX_OTHER]
    for r in cry: samples.append((os.path.join(audio, r["filename"]), "baby_cry"))
    for r in oth: samples.append((os.path.join(audio, r["filename"]), "other"))
if DONATEACRY_ROOT and os.path.isdir(DONATEACRY_ROOT):
    for p in _list_audio(DONATEACRY_ROOT): samples.append((p, "baby_cry"))
if SCREAM_ROOT and os.path.isdir(SCREAM_ROOT):
    for p in _list_audio(SCREAM_ROOT): samples.append((p, "scream"))
assert samples, "No audio found - check ESC50_ROOT / DONATEACRY_ROOT."

CLASS_NAMES = sorted({l for _, l in samples})
LABEL2IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
print("Classes:", CLASS_NAMES, "| clips:", len(samples))

def logmel(path):
    try:
        y, _ = librosa.load(path, sr=SR, mono=True)
    except Exception:
        return None
    if len(y) == 0: return None
    if len(y) < N_SAMPLES: y = np.pad(y, (0, N_SAMPLES - len(y)))
    else: y = y[:N_SAMPLES]
    m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
    m = librosa.power_to_db(m, ref=np.max)
    m = (m - m.mean()) / (m.std() + 1e-6)
    return m.astype(np.float32)

def cache_path(p):
    h = hashlib.md5(p.encode()).hexdigest()[:16]
    return os.path.join(CACHE_DIR, os.path.splitext(os.path.basename(p))[0] + "_" + h + ".npy")

manifest = []
for p, lab in tqdm(samples, desc="Caching log-mel"):
    cp = cache_path(p)
    if not os.path.exists(cp):
        feat = logmel(p)
        if feat is None: continue
        np.save(cp, feat)
    manifest.append({"npy": cp, "label": LABEL2IDX[lab]})
import json
json.dump({"class_names": CLASS_NAMES, "items": manifest}, open(os.path.join(WORK_DIR, "manifest.json"), "w"))
print("Cached:", len(manifest))

## Phase 2 - Datasets / loaders (class-balanced + light SpecAugment)

In [ ]:
import json, numpy as np, torch, random
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
blob = json.load(open(os.path.join(WORK_DIR, "manifest.json")))
CLASS_NAMES = blob["class_names"]; manifest = blob["items"]; NUM_CLASSES = len(CLASS_NAMES)
random.seed(SEED); np.random.seed(SEED)

# Pool all clips, then a stratified 75/25 split: 75% train+val pool (K-fold CV
# runs here) and a 25% held-out TEST set (scored once in Evaluate).
all_items  = list(manifest)
all_labels = np.array([m["label"] for m in all_items])
trainval_items, test_items = train_test_split(
    all_items, test_size=0.25, random_state=SEED, stratify=all_labels)
print("Classes:", NUM_CLASSES, "| pool:", len(all_items),
      "| train+val:", len(trainval_items), " held-out test:", len(test_items))

def spec_augment(m):
    m = m.copy()
    if random.random() < 0.5:
        f = random.randint(0, 8); f0 = random.randint(0, max(0, m.shape[0]-f)); m[f0:f0+f, :] = 0
    if random.random() < 0.5:
        t = random.randint(0, 16); t0 = random.randint(0, max(0, m.shape[1]-t)); m[:, t0:t0+t] = 0
    return m

class MelDataset(Dataset):
    def __init__(self, items, augment=False): self.items = items; self.augment = augment
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        m = np.load(self.items[i]["npy"]).astype(np.float32)
        if self.augment: m = spec_augment(m)
        return torch.from_numpy(m).unsqueeze(0), self.items[i]["label"]

def make_loaders(tr_items, va_items):
    tr_ds = MelDataset(tr_items, augment=True); va_ds = MelDataset(va_items)
    labels = np.array([m["label"] for m in tr_items])
    cls_count = np.bincount(labels, minlength=NUM_CLASSES).astype(np.float32)
    cls_w = 1.0 / np.maximum(cls_count, 1.0); samp_w = cls_w[labels]
    sampler = WeightedRandomSampler(samp_w, num_samples=len(tr_items), replacement=True)
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return tr_loader, va_loader, cls_w

_cnt = np.bincount(np.array([m["label"] for m in trainval_items]), minlength=NUM_CLASSES).astype(int)
for c, n in zip(CLASS_NAMES, _cnt): print("  %-10s %d" % (c, n))

## Model - compact 2D CNN over log-mel

In [ ]:
import torch.nn as nn
class AudioCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def blk(i, o): return nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.net = nn.Sequential(blk(1, 32), blk(32, 64), blk(64, 128), blk(128, 128))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(128, n_classes))
    def forward(self, x): return self.head(self.net(x))

_m = AudioCNN(NUM_CLASSES); _d = torch.zeros(2, 1, N_MELS, 1 + N_SAMPLES // HOP)
print("Model OK, output:", tuple(_m(_d).shape))

## Train - Stratified 5-Fold CV (best model by val_loss)
Stratified K-Fold on the 75% train+val pool; each fold checkpoints on the
**lowest val_loss** and early-stops on patience. The best-val_loss fold is saved;
the 25% held-out test is scored next. High cry recall + high `other` precision
is the goal.

In [ ]:
import os, copy, numpy as np, torch, torch.nn as nn
from sklearn.model_selection import StratifiedKFold
device = "cuda" if torch.cuda.is_available() else "cpu"
N_SPLITS = 5            # Stratified K-Fold on the 75% train+val pool
PATIENCE = 8            # early stop after this many epochs without val_loss gain
save_path = os.path.join(WORK_DIR, "audio_event_cnn_best.pt")

def run_eval(model, loader, criterion):
    model.eval(); vloss = 0.0; vc = vn = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = criterion(out, y)
            vloss += loss.item() * y.size(0)
            vc += (out.argmax(1) == y).sum().item(); vn += y.size(0)
    return vloss / max(vn, 1), vc / max(vn, 1)

def train_one_fold(tr_items, va_items, fold):
    tr_loader, va_loader, cls_w = make_loaders(tr_items, va_items)
    model = AudioCNN(NUM_CLASSES).to(device)
    w = torch.tensor(cls_w / cls_w.sum() * NUM_CLASSES, dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None
    best_vloss = float("inf"); best_va = 0.0; best_state = None; bad = 0
    for epoch in range(EPOCHS):
        model.train(); tc = tn = 0; run = 0.0
        for x, y in tr_loader:
            x, y = x.to(device), y.to(device); opt.zero_grad()
            if scaler:
                with torch.amp.autocast("cuda"):
                    out = model(x); loss = criterion(out, y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                out = model(x); loss = criterion(out, y); loss.backward(); opt.step()
            run += loss.item() * y.size(0); tc += (out.argmax(1) == y).sum().item(); tn += y.size(0)
        sched.step()
        vloss, va = run_eval(model, va_loader, criterion)
        print("  [fold %d] ep %02d/%d  loss=%.4f train_acc=%.3f  val_loss=%.4f val_acc=%.3f" % (
            fold, epoch + 1, EPOCHS, run / max(tn, 1), tc / max(tn, 1), vloss, va))
        if vloss < best_vloss - 1e-4:
            best_vloss = vloss; best_va = va; best_state = copy.deepcopy(model.state_dict()); bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  [fold %d] early stop @ epoch %d (best val_loss=%.4f)" % (fold, epoch + 1, best_vloss))
                break
    return best_state, best_vloss, best_va

y_pool = np.array([m["label"] for m in trainval_items])
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_vloss, fold_vacc = [], []
global_best_vloss = float("inf"); global_best_state = None
for fold, (tr_idx, va_idx) in enumerate(skf.split(trainval_items, y_pool), 1):
    tr_items = [trainval_items[i] for i in tr_idx]
    va_items = [trainval_items[i] for i in va_idx]
    print("Fold %d/%d  train=%d  val=%d" % (fold, N_SPLITS, len(tr_items), len(va_items)))
    state, vloss, va = train_one_fold(tr_items, va_items, fold)
    fold_vloss.append(vloss); fold_vacc.append(va)
    if vloss < global_best_vloss:
        global_best_vloss = vloss; global_best_state = state
        print("  ** new global best val_loss=%.4f (fold %d) **" % (vloss, fold))

print("\nCV val_loss: %.4f +/- %.4f" % (float(np.mean(fold_vloss)), float(np.std(fold_vloss))))
print("CV val_acc : %.4f +/- %.4f" % (float(np.mean(fold_vacc)), float(np.std(fold_vacc))))
torch.save({"model_state_dict": global_best_state, "class_names": CLASS_NAMES,
    "val_loss": float(global_best_vloss),
    "cv_val_loss_mean": float(np.mean(fold_vloss)), "cv_val_acc_mean": float(np.mean(fold_vacc)),
    "config": {"sr": SR, "duration": DURATION, "n_fft": N_FFT, "hop": HOP,
        "n_mels": N_MELS, "normalize": "per_clip_zscore"}}, save_path)
print("Saved best-fold model (val_loss=%.4f) ->" % global_best_vloss, save_path)

## Evaluate - per-class precision / recall
For cry detection you want high **recall** (don't miss a real cry) while keeping
`other` precision high (don't false-alarm on TV / talking).

In [ ]:
import numpy as np, torch
from torch.utils.data import DataLoader
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(os.path.join(WORK_DIR, "audio_event_cnn_best.pt"), map_location=device)
model = AudioCNN(NUM_CLASSES).to(device)
model.load_state_dict(ckpt["model_state_dict"]); model.eval()
test_loader = DataLoader(MelDataset(test_items, augment=False),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
with torch.no_grad():
    for x, y in test_loader:
        p = model(x.to(device)).argmax(1).cpu().numpy(); y = y.numpy()
        for t, pp in zip(y, p): cm[t, pp] += 1
print("HELD-OUT TEST (25%)  -  n =", int(cm.sum()))
print("%-10s %6s %6s %6s" % ("class", "prec", "rec", "n"))
for i, c in enumerate(CLASS_NAMES):
    tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i, :].sum() - tp
    print("%-10s %6.3f %6.3f %6d" % (c, tp / max(tp + fp, 1), tp / max(tp + fn, 1), cm[i, :].sum()))
print("\nTest acc:", round(float(np.trace(cm)) / max(int(cm.sum()), 1), 4),
      "| CV val_loss mean:", round(ckpt["cv_val_loss_mean"], 4))

## Export & wire back
`audio_event_cnn_best.pt` is in `WORK_DIR` on Drive.
1. Download it into the repo at `models/weights/audio_event_cnn_best.pt`.
2. The checkpoint carries `class_names` + the exact front-end (`sr`, `n_fft`,
   `hop`, `n_mels`, `duration`) so inference recomputes identical features.
3. Ping me and I'll replace the DSP cry detector in `stages/audio_analysis.py`:
   sliding `DURATION`-second window -> same log-mel -> this CNN -> per-class
   probabilities, with temporal persistence before raising a cry/scream event.

**Add scream:** drop a scream dataset into `SCREAM_ROOT` and re-run; the class is
auto-added and wired the same way.